# التقسيط في السوق المصري — التنضيف والاستكشاف (EDA)
**السؤال:** لما تقسط حاجة في مصر، بتدفع كام زيادة فعلًا؟ وأنهي قطاع وأنهي شركة أغلى؟

الداتا: `egypt_installments_offers.csv` (عروض بالأرقام) + `egypt_installments_plan_terms.csv` (شروط الشركات) — مجمعة يدويًا من مواقع السوق المصري في أكتوبر 2026، وكل صف معاه رابط مصدره.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

offers = pd.read_csv("egypt_installments_offers.csv")
terms  = pd.read_csv("egypt_installments_plan_terms.csv")
print(offers.shape, terms.shape)
offers.head()

## 1) فحص الداتا

In [ ]:
offers.info()

In [ ]:
# القيم الناقصة — مفهومة: العقارات مفيهاش سعر كاش، والسيناريوهات مفيهاش listed price
offers.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
# أنواع البيانات: الحقيقي مقابل المحسوب
offers["data_type"].value_counts()

## 2) التنضيف
- أسماء إنجليزي للقطاعات والشركات (علشان الرسومات تتقري صح — matplotlib مش بيعرض العربي كويس)
- عمود `is_real`: العروض المنشورة فعلًا مقابل السيناريوهات
- عمود `is_zero_interest`: هل العرض فعلًا بدون أي زيادة؟
- حساب الفايدة لنفسنا من الصفر (للتأكد من أرقام الإكسيل)

In [ ]:
sector_en = {
 "موبايلات وإلكترونيات":"Phones & Electronics","أجهزة منزلية":"Home Appliances","أثاث ومطابخ":"Furniture",
 "عربيات":"Cars","عقارات":"Real Estate","دهب وشبكة":"Gold","تعليم":"Education","سفر وعمرة وحج":"Travel & Umrah",
 "علاج وأسنان":"Medical & Dental","لبس وموضة":"Fashion (BNPL)","موتوسيكلات وتكاتك":"Motorcycles",
 "مستحضرات تجميل وعطور":"Beauty","اشتراكات جيم ونوادي":"Gyms & Clubs","طاقة شمسية وتكييفات":"Solar & AC",
 "تشطيبات وأدوات صحية":"Finishing","قاعات أفراح وتجهيز عرايس":"Weddings","ألعاب وبلايستيشن":"Gaming",
 "عجل ومستلزمات رياضية":"Bicycles","تأمين":"Insurance"}

def fin_en(x):
    x = str(x)
    for k, v in [("موجو","MOGO"),("أمان","Aman"),("InBank","InBank"),("TRU","TRU"),("فرصة","Forsa"),("سيمبل","Sympl"),
                 ("البنك الأهلي","NBE"),("بنك مصر","Banque Misr"),("بطاقات البنوك","Bank cards"),("بريميوم","Premium Card"),
                 ("تقسيط بنكي","Bank (auto)"),("النادي","Club (direct)"),("العيادة","Clinic (direct)"),("المطور","Developer")]:
        if k in x: return v
    return "Other"

df = offers.copy()
df["sector_en"]    = df["sector"].map(sector_en)
df["financier_en"] = df["financier"].map(fin_en)
df["is_real"]      = ~df["data_type"].str.contains("سيناريو")
df["has_rate"]     = df["apr_effective"].notna()
df["is_zero_interest"] = df["extra_pct"].abs() < 0.005
df[["sector","sector_en","financier","financier_en","is_real"]].drop_duplicates("sector").head(20)

In [ ]:
# نتحقق من الفايدة بنفسنا: نحل معادلة القسط بالـ bisection (من غير مكتبات خارجية)
def monthly_rate(pv, pmt, n):
    if pmt * n <= pv + 0.5: return 0.0
    lo, hi = 0.0, 1.0
    for _ in range(200):
        r = (lo + hi) / 2
        if pmt * (1 - (1 + r) ** -n) / r > pv: lo = r
        else: hi = r
    return (lo + hi) / 2

simple = df[df.has_rate & (df.annual_payments_total.fillna(0) == 0)].copy()
pv = simple.cash_price - simple.down_payment.fillna(0) - simple.upfront_fees.fillna(0)
simple["apr_check"] = [(1 + monthly_rate(p, m, int(n))) ** 12 - 1 for p, m, n in zip(pv, simple.monthly_installment, simple.months)]
print("أكبر فرق بين حسابنا وحساب الملف:", (simple.apr_check - simple.apr_effective).abs().max().round(5))

## 3) الصورة العامة

In [ ]:
rated = df[df.has_rate]
kpi = pd.Series({
    "عدد العروض": len(df),
    "عروض بفايدة محسوبة": len(rated),
    "عدد القطاعات": df.sector.nunique(),
    "عروض بدون أي زيادة": int(rated.is_zero_interest.sum()),
    "متوسط الزيادة عن الكاش": f"{rated.extra_pct.mean():.1%}",
    "متوسط الفايدة السنوية الفعلية (للعروض اللي فيها فايدة)": f"{rated.loc[~rated.is_zero_interest,'apr_effective'].mean():.1%}",
})
kpi

In [ ]:
cov = df.groupby("sector_en").agg(offers=("id","count"), real=("is_real","sum")).sort_values("offers")
cov.plot.barh(figsize=(8,6), color=["#4C72B0","#55A868"])
plt.title("Rows per sector (total vs. real published offers)"); plt.xlabel("rows"); plt.ylabel("")
plt.tight_layout(); plt.show()

## 4) أنهي قطاع أغلى في التقسيط؟

In [ ]:
by_sector = (rated.groupby("sector_en")
             .agg(n=("id","count"), avg_extra=("extra_pct","mean"), avg_apr=("apr_effective","mean"),
                  zero_share=("is_zero_interest","mean"))
             .sort_values("avg_apr", ascending=False))
by_sector.style.format({"avg_extra":"{:.1%}","avg_apr":"{:.1%}","zero_share":"{:.0%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
by_sector["avg_apr"].sort_values().plot.barh(ax=ax, color="#C44E52")
ax.axvline(0.128, ls="--", c="gray");  ax.text(0.13, 0.2, "Inflation 12.8%", color="gray")
ax.axvline(0.1775, ls=":", c="green"); ax.text(0.18, 1.2, "Bank CD 17.75%", color="green")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
ax.set_title("Average effective annual interest by sector"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 5) أنهي شركة تقسيط أغلى؟

In [ ]:
by_fin = (rated[~rated.is_zero_interest].groupby("financier_en")
          .agg(n=("id","count"), avg_apr=("apr_effective","mean"), max_apr=("apr_effective","max"))
          .sort_values("avg_apr", ascending=False))
by_fin.style.format({"avg_apr":"{:.1%}","max_apr":"{:.1%}"})

In [ ]:
plot_df = rated[~rated.is_zero_interest & rated.financier_en.isin(by_fin[by_fin.n >= 2].index)]
fig, ax = plt.subplots(figsize=(8,4.5))
sns.boxplot(data=plot_df, y="financier_en", x="apr_effective", order=by_fin[by_fin.n>=2].index, ax=ax, color="#8172B2")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
ax.set_title("Effective annual interest by financier (offers with any cost)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 6) المدة بتفرق قد إيه؟

In [ ]:
dur = rated[rated.months.between(3, 36)]
fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.scatterplot(data=dur, x="months", y="extra_pct", hue="financier_en", ax=axes[0], s=60)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
axes[0].set_title("Extra paid vs. duration"); axes[0].legend(fontsize=7)
sns.scatterplot(data=dur, x="months", y="apr_effective", hue="financier_en", ax=axes[1], s=60, legend=False)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
axes[1].set_title("Effective annual rate vs. duration")
plt.tight_layout(); plt.show()

**ملاحظة:** الزيادة الإجمالية بتطلع مع المدة بشكل خطي تقريبًا، لكن الفايدة السنوية الفعلية بتفضل ثابتة تقريبًا لكل شركة —
يعني المدة الأطول مش "أرخص"، هي بس بتوزع نفس السعر على وقت أطول.

## 7) خدعة "بدون فوائد": المصاريف الإدارية

In [ ]:
fees = rated[(rated.upfront_fees.fillna(0) > 0)].copy()
fees["fee_pct"] = fees.upfront_fees / fees.cash_price
fees[["item","financier_en","months","fee_pct","apr_effective"]].sort_values("apr_effective", ascending=False) \
    .style.format({"fee_pct":"{:.1%}","apr_effective":"{:.1%}"})

عرض مكتوب عليه **"بدون فوائد"** بس فيه مصاريف إدارية 10% على 6 شهور = فايدة فعلية حوالي **44% في السنة**،
لأن المصاريف بتتدفع مرة واحدة على فترة قصيرة.

## 8) هل التقسيط بيكسب التضخم؟

In [ ]:
inflation, cd_rate = 0.128, 0.1775
rated = rated.assign(vs_cd=np.select(
    [rated.apr_effective <= 0.0001, rated.apr_effective < cd_rate],
    ["0% — قسّط وحط الفلوس في شهادة", "أقل من عائد الشهادة"], "أغلى من عائد الشهادة — ادفع كاش"))
rated["vs_cd"].value_counts()

## 9) العقارات: الفايدة مدفونة في السعر

In [ ]:
re_ = df[df.sector_en == "Real Estate"][["item","listed_installment_price","down_payment","months","monthly_installment","total_paid","diff_vs_listed_price"]]
re_["down_pct"] = re_.down_payment / re_.listed_installment_price
re_

سعر الإعلان = إجمالي الأقساط بالظبط في أغلب الحالات، والسعر الكاش مش منشور. يعني مش ممكن نحسب الفايدة من الإعلان لوحده —
محتاجين نسأل المطور عن **خصم الكاش**. (ونفس الكلام لصف 177م اللي إجمالي أقساطه أقل من السعر بنص مليون — غالبًا فيه دفعة استلام مش مكتوبة.)

## 10) الدهب

In [ ]:
terms[terms.sector == "دهب وشبكة"][["financier","interest","notes","date"]]

## 11) أهم النتائج
1. **الأثاث** أغلى قطاع في التقسيط: موجو حوالي 35% سنويًا، وأمان حوالي 43–46% — وده ثابت على كل المنتجات.
2. **الموبايلات** أرخص قطاع: موقع سامسونج الرسمي 24 شهر بدون فوائد فعلًا، لكن نفس الموبايل في بي تك أغلى بـ25% كاش.
3. **"بدون فوائد"** مع مصاريف إدارية ممكن تساوي 30–44% فايدة سنوية.
4. **المدة** بتزود الإجمالي لكن مش بتغير الفايدة السنوية كتير.
5. **العقارات:** السعر المعلن هو سعر التقسيط، والفايدة مش ظاهرة.
6. **الدهب:** تمويله بالتقسيط اتمنع من 16 سبتمبر 2026.

**حدود الداتا:** التجميل والطاقة الشمسية والأفراح والتأمين فيهم شروط بس من غير أسعار؛ وبعض الصفوف "سيناريو" (سعر حقيقي × شروط منشورة).

In [ ]:
# حفظ النسخة النضيفة لـ SQL و Power BI
df.to_csv("offers_clean.csv", index=False, encoding="utf-8-sig")
print("saved offers_clean.csv", df.shape)